###  Text Preprocessing Pipeline
A complete preprocessing system that takes raw text input and outputs clean, structured text ready for machine learning models.
This pipeline should handle:
1. Cleaning raw text
2. Custom tokenization (not just default library use)
3. Stopword removal
4. Lemmatization
5. Output formatted processed text


In [337]:
# import the required libraries for this project
# note: we will be using Spacy instead of NLTK beacuse of its efficiency, speed and ease of use for NLP tasks
import spacy
import re
import string
import pandas as pd
import numpy as np
from nltk.stem import PorterStemmer # For the bonus comparison
from nltk.tokenize import word_tokenize # For the bonus comparison

#remove warnings
import warnings
warnings.filterwarnings('ignore')

In [338]:
# contractions dictionary, we will use this to expand contractions in the text, like "don't" to "do not", "can't" to "cannot"
contractions_dict = {
    "I'm": "I am",
    "I'm'a": "I am about to",
    "I'm'o": "I am going to",
    "I've": "I have",
    "I'll": "I will",
    "I'll've": "I will have",
    "I'd": "I would",
    "I'd've": "I would have",
    "Whatcha": "What are you",
    "amn't": "am not",
    "ain't": "are not",
    "aren't": "are not",
    "'cause": "because",
    "can't": "cannot",
    "can't've": "cannot have",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "daren't": "dare not",
    "daresn't": "dare not",
    "dasn't": "dare not",
    "didn't": "did not",
    "didn’t": "did not",
    "don't": "do not",
    "don’t": "do not",
    "doesn't": "does not",
    "e'er": "ever",
    "everyone's": "everyone is",
    "finna": "fixing to",
    "gimme": "give me",
    "gon't": "go not",
    "gonna": "going to",
    "gotta": "got to",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "he've": "he have",
    "he's": "he is",
    "he'll": "he will",
    "he'll've": "he will have",
    "he'd": "he would",
    "he'd've": "he would have",
    "here's": "here is",
    "how're": "how are",
    "how'd": "how did",
    "how'd'y": "how do you",
    "how's": "how is",
    "how'll": "how will",
    "isn't": "is not",
    "it's": "it is",
    "'tis": "it is",
    "'twas": "it was",
    "it'll": "it will",
    "it'll've": "it will have",
    "it'd": "it would",
    "it'd've": "it would have",
    "kinda": "kind of",
    "let's": "let us",
    "luv": "love",
    "ma'am": "madam",
    "may've": "may have",
    "mayn't": "may not",
    "might've": "might have",
    "mightn't": "might not",
    "mightn't've": "might not have",
    "must've": "must have",
    "mustn't": "must not",
    "mustn't've": "must not have",
    "needn't": "need not",
    "needn't've": "need not have",
    "ne'er": "never",
    "o'": "of",
    "o'clock": "of the clock",
    "ol'": "old",
    "oughtn't": "ought not",
    "oughtn't've": "ought not have",
    "o'er": "over",
    "shan't": "shall not",
    "sha'n't": "shall not",
    "shalln't": "shall not",
    "shan't've": "shall not have",
    "she's": "she is",
    "she'll": "she will",
    "she'd": "she would",
    "she'd've": "she would have",
    "should've": "should have",
    "shouldn't": "should not",
    "shouldn't've": "should not have",
    "so've": "so have",
    "so's": "so is",
    "somebody's": "somebody is",
    "someone's": "someone is",
    "something's": "something is",
    "sux": "sucks",
    "that're": "that are",
    "that's": "that is",
    "that'll": "that will",
    "that'd": "that would",
    "that'd've": "that would have",
    "'em": "them",
    "there're": "there are",
    "there's": "there is",
    "there'll": "there will",
    "there'd": "there would",
    "there'd've": "there would have",
    "these're": "these are",
    "they're": "they are",
    "they've": "they have",
    "they'll": "they will",
    "they'll've": "they will have",
    "they'd": "they would",
    "they'd've": "they would have",
    "this's": "this is",
    "this'll": "this will",
    "this'd": "this would",
    "those're": "those are",
    "to've": "to have",
    "wanna": "want to",
    "wasn't": "was not",
    "we're": "we are",
    "we've": "we have",
    "we'll": "we will",
    "we'll've": "we will have",
    "we'd": "we would",
    "we'd've": "we would have",
    "weren't": "were not",
    "what're": "what are",
    "what'd": "what did",
    "what've": "what have",
    "what's": "what is",
    "what'll": "what will",
    "what'll've": "what will have",
    "when've": "when have",
    "when's": "when is",
    "where're": "where are",
    "where'd": "where did",
    "where've": "where have",
    "where's": "where is",
    "which's": "which is",
    "who're": "who are",
    "who've": "who have",
    "who's": "who is",
    "who'll": "who will",
    "who'll've": "who will have",
    "who'd": "who would",
    "who'd've": "who would have",
    "why're": "why are",
    "why'd": "why did",
    "why've": "why have",
    "why's": "why is",
    "will've": "will have",
    "won't": "will not",
    "won't've": "will not have",
    "would've": "would have",
    "wouldn't": "would not",
    "wouldn't've": "would not have",
    "y'all": "you all",
    "y'all're": "you all are",
    "y'all've": "you all have",
    "y'all'd": "you all would",
    "y'all'd've": "you all would have",
    "you're": "you are",
    "you've": "you have",
    "you'll've": "you shall have",
    "you'll": "you will",
    "you'd": "you would",
    "you'd've": "you would have",

    "to cause": "to cause",
    "will cause": "will cause",
    "should cause": "should cause",
    "would cause": "would cause",
    "can cause": "can cause",
    "could cause": "could cause",
    "must cause": "must cause",
    "might cause": "might cause",
    "shall cause": "shall cause",
    "may cause": "may cause"
}

In [339]:
# the first step is to remove contractions, like "don't" to "do not", "can't" to "cannot"
def expand_contractions(text):
    for word, replacement in contractions_dict.items():
        text = text.replace(word, replacement)
    return text

In [340]:
# step 1: clean the text data by removing special characters, numbers, punctuation, and lowercasing the text
def clean_text(text):

    # Handle empty or invalid input
    if not isinstance(text, str):
        return ""
    
    # convert to lowercase
    text = text.lower()

    # remove special characters and numbers
    text = re.sub(r'[^a-z0-9\s\.]', ' ', text)

     # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [341]:
# Step 2: creating a coustom tokenizer using regex to split the text into tokens (words)
def custom_tokenizer(text):
    
    # Handle empty or invalid input
    if not isinstance(text, str):
        return []

    # STEP 1: Tokenize properly
    # - \d+\.\d+  → 100.00 (decimal numbers stay intact)
    # - \b\w+\b    → words like preferred, this
    # - [^\w\s]    → punctuation like .
    
    tokens = re.findall(r'\d+\.\d+|\b\w+\b|[^\w\s]', text)
    
    return tokens

In [342]:
# we will remove punctuation here, its best here because it can creates problems later on real data.
def remove_punctuation(tokens):
    return [t for t in tokens if re.match(r'\w+|\d+\.\d+', t)]

In [343]:
# step 3: remove stopwords using nltk stopwords.
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [t for t in tokens if t.lower() not in stop_words]

In [344]:
# step 4: lemmatization using spacy
nlp = spacy.load("en_core_web_sm")
def lemmatize_text(tokens):
    """Converts words into base form:
    running → run
    was → be
    better → good (context aware)
    """

    doc = nlp(" ".join(tokens))
    return [token.lemma_ for token in doc]


In [345]:
# Integrating all the steps into a single function to create a text processing pipeline
def processing_text(text):
    # Handle empty or invalid input
    if not isinstance(text, str):
        return []
    
    # used for dealing with contractions, like "don't" to "do not", "can't" to "cannot".
    text = expand_contractions(text)

    # cleaning the text by removing special characters, and lowercasing the text.
    text = clean_text(text)

    # tokenizing the text into words using a custom tokenizer.
    tokens = custom_tokenizer(text)
    
    # removing punctuation from the tokens, its best here because it can creates problems later on real data.
    tokens = remove_punctuation(tokens)

    # removing stopwords from the tokens.
    tokens = remove_stopwords(tokens)

    # lemmatizing the tokens using spacy.
    lemmatized_tokens = lemmatize_text(tokens)
    return lemmatized_tokens

In [346]:
# checking the entire pipeline and comapring it with raw text, to see if its working fine or not.
def show_raw_vs_processed(text):
    print("RAW TEXT:")
    print(text)

    processed = preprocess_text(text)

    print("\nPROCESSED TEXT:")
    print(processed)

In [347]:
sample_text = "A sample it's, this wasn't text conatins $100.0, playing, crying, preferred, some $$#^&$^*7.,;, and yeah trying to work better for beautiful cleaned text."
show_raw_vs_processed(sample_text)

RAW TEXT:
A sample it's, this wasn't text conatins $100.0, playing, crying, preferred, some $$#^&$^*7.,;, and yeah trying to work better for beautiful cleaned text.

PROCESSED TEXT:
['sample', 'text', 'conatin', '100.0', 'playing', 'cry', 'prefer', '7', 'yeah', 'try', 'work', 'well', 'beautiful', 'clean', 'text']


In [348]:
def preprocess_with_logs(text):
    print("STEP 1: INPUT")
    print(text)

    text = expand_contractions(text)
    print("STEP 2: AFTER CONTRACTIONS")
    print(text)

    text = clean_text(text)
    print("STEP 3: CLEANED")
    print(text)

    tokens = custom_tokenizer(text)
    print("STEP 4: TOKENS")
    print(tokens)

    tokens = remove_stopwords(tokens)
    print("STEP 5: AFTER STOPWORDS")
    print(tokens)

    final = lemmatize_text(tokens)
    print("STEP 6: FINAL OUTPUT")
    print(final)

    return final

In [349]:
sample = "I've been running, trying, ^*&$*,, tried enjoyed and playing with better tools"
print(preprocess_with_logs(sample))

STEP 1: INPUT
I've been running, trying, ^*&$*,, tried enjoyed and playing with better tools
STEP 2: AFTER CONTRACTIONS
I have been running, trying, ^*&$*,, tried enjoyed and playing with better tools
STEP 3: CLEANED
i have been running trying tried enjoyed and playing with better tools
STEP 4: TOKENS
['i', 'have', 'been', 'running', 'trying', 'tried', 'enjoyed', 'and', 'playing', 'with', 'better', 'tools']
STEP 5: AFTER STOPWORDS
['running', 'trying', 'tried', 'enjoyed', 'playing', 'better', 'tools']
STEP 6: FINAL OUTPUT
['run', 'try', 'try', 'enjoy', 'play', 'well', 'tool']
['run', 'try', 'try', 'enjoy', 'play', 'well', 'tool']


In [350]:
# comparing stemming vs lemmatization using your pipeline

from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def compare_stem_vs_lemma(text):
    """
    Compare stemming vs lemmatization using your pipeline
    """

    print("\n==============================")
    print("🔹 RAW TEXT")
    print("==============================")
    print(text)

    # ---------------- LEMMATIZATION ----------------
    lemma = processing_text(text)

    # ---------------- STEMMING PIPELINE ----------------
    text_clean = expand_contractions(text)
    text_clean = clean_text(text_clean)
    tokens = custom_tokenizer(text_clean)
    tokens = remove_punctuation(tokens)
    tokens = remove_stopwords(tokens)

    stemmed = [stemmer.stem(t) for t in tokens]

    print("\n==============================")
    print("🔹 STEMMED OUTPUT")
    print("==============================")
    print(stemmed)

    print("\n==============================")
    print("🔹 LEMMATIZED OUTPUT")
    print("==============================")
    print(lemma)

    return {
        "stemmed": stemmed,
        "lemmatized": lemma
    }

In [351]:
sample_text = "I've been running, trying, ^*&$*,, tried enjoyed and playing with better tools" 
compare_stem_vs_lemma(sample_text)


🔹 RAW TEXT
I've been running, trying, ^*&$*,, tried enjoyed and playing with better tools

🔹 STEMMED OUTPUT
['run', 'tri', 'tri', 'enjoy', 'play', 'better', 'tool']

🔹 LEMMATIZED OUTPUT
['run', 'try', 'try', 'enjoy', 'play', 'well', 'tool']


{'stemmed': ['run', 'tri', 'tri', 'enjoy', 'play', 'better', 'tool'],
 'lemmatized': ['run', 'try', 'try', 'enjoy', 'play', 'well', 'tool']}

In [352]:
# batch processing of multiple texts using the pipeline and also to get data from csv files.
def process_dataset(file_path, text_column, output_file="processed_output.csv"):
    """
    Process entire dataset and save cleaned output
    """

    df = pd.read_csv(file_path)

    processed_texts = []

    for text in df[text_column].fillna(""):
        processed = processing_text(text)
        processed_texts.append(" ".join(processed))

    df["processed_text"] = processed_texts

    df.to_csv(output_file, index=False)

    print("\n✅ Dataset processed successfully!")
    print("Saved as:", output_file)

    return df

In [353]:
# creating a functiion for cli interface to process text files in batch using the pipeline, and also to get data from csv files.
def run_cli():
    """
    Simple CLI menu for your NLP pipeline
    """

    while True:
        print("\n==============================")
        print(" NLP PIPELINE MENU")
        print("==============================")
        print("1. Process Single Text")
        print("2. Compare Stem vs Lemma")
        print("3. Process Dataset (CSV)")
        print("4. Exit")

        choice = input("Enter choice: ")

        if choice == "1":
            text = input("Enter text: ")
            print(processing_text(text))

        elif choice == "2":
            text = input("Enter text: ")
            compare_stem_vs_lemma(text)

        elif choice == "3":
            path = input("Enter CSV file path: ")
            col = input("Enter text column name: ")
            process_dataset(path, col)

        elif choice == "4":
            print("Exiting...")
            break

        else:
            print("Invalid choice. Try again.")